# 96. The Resilient Network Design (Robust Optimization)
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- We have a set of potential facility locations and demand zones
- Multiple uncertainty scenarios represent different disruption events
- Facilities can fail completely or operate at reduced capacity
- Transportation routes can be blocked during disruptions
- Demand patterns shift significantly during disruption scenarios

### Approach (step-by-step)
1. **Formulate the robust optimization problem** as a min-max optimization
2. **Define first-stage decisions** (facility location variables)
3. **Define second-stage decisions** (flow variables for each scenario)
4. **Implement mixed-integer programming** with scenario constraints
5. **Solve using pulp** and analyze the worst-case performance

### What to look for in the results
- Facility selection that minimizes worst-case total cost
- Service coverage levels across all scenarios
- Trade-offs between investment cost and resilience
- Sensitivity to different parameter values

### Concrete example (from the source)
Consider a simplified network with 3 potential facilities and 3 demand zones:
- **Normal scenario**: All facilities operational, demands = [100, 150, 200]
- **Hurricane scenario**: Facility 2 fails, route 1-3 blocked, demands = [300, 50, 100]
- **Objective**: Select facilities to minimize worst-case cost while maintaining 80% service coverage

In [1]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [2]:
# Define problem data structures
class Facility:
    def __init__(self, id, x, y, capacity, fixed_cost):
        self.id = id
        self.x = x
        self.y = y
        self.capacity = capacity
        self.fixed_cost = fixed_cost

class DemandZone:
    def __init__(self, id, x, y, demand):
        self.id = id
        self.x = x
        self.y = y
        self.demand = demand

class Scenario:
    def __init__(self, id, probability, facility_failures, route_blocks, demand_multipliers):
        self.id = id
        self.probability = probability
        self.facility_failures = facility_failures  # List of facility IDs that fail
        self.route_blocks = route_blocks  # List of (fac_id, demand_id) tuples that are blocked
        self.demand_multipliers = demand_multipliers  # Dict mapping demand_id to multiplier

In [ ]:
# Create the concrete example from the source
# Define facilities
facilities = [
    Facility(1, 10, 50, 800, 500000),  # Location 1
    Facility(2, 50, 40, 600, 400000),  # Location 2 (central facility)
    Facility(3, 20, 70, 900, 600000),  # Location 3
]

# Define demand zones
demand_zones = [
    DemandZone(1, 15, 45, 100),  # Zone 1
    DemandZone(2, 35, 20, 150),  # Zone 2
    DemandZone(3, 25, 40, 200),  # Zone 3
]

# Define scenarios
scenarios = [
    # Normal scenario (Scenario 1)
    Scenario(
        id=1,
        probability=0.6,
        facility_failures=[],  # All facilities operational
        route_blocks=[],  # All routes open
        demand_multipliers={1: 1.0, 2: 1.0, 3: 1.0}  # Normal demands
    ),
    # Hurricane scenario (Scenario 2)
    Scenario(
        id=2,
        probability=0.4,
        facility_failures=[2],  # Facility 2 fails
        route_blocks=[(1, 3)],  # Route from facility 1 to demand zone 3 blocked
        demand_multipliers={1: 3.0, 2: 0.33, 3: 0.5}  # Shifted demands
    )
]

print(f"Problem Setup:")
print(f"- Facilities: {len(facilities)} potential locations")
print(f"- Demand zones: {len(demand_zones)} service areas")
print(f"- Scenarios: {len(scenarios)} uncertainty scenarios")
print(f"- Service requirement: At least 80% coverage in worst case")

In [ ]:
# Calculate transportation costs between facilities and demand zones
def calculate_distance(facility, demand_zone):
    """Calculate Euclidean distance between facility and demand zone"""
    return np.sqrt((facility.x - demand_zone.x)**2 + (facility.y - demand_zone.y)**2)

def calculate_transport_cost(facility, demand_zone, quantity):
    """Calculate transport cost based on distance and quantity"""
    distance = calculate_distance(facility, demand_zone)
    return distance * 0.1 * quantity  # $0.1 per unit distance per unit demand

# Create cost matrix for each scenario
transport_costs = {}
for scenario in scenarios:
    costs = np.zeros((len(facilities), len(demand_zones)))
    for i, facility in enumerate(facilities):
        for j, demand in enumerate(demand_zones):
            if facility.id in scenario.facility_failures:
                costs[i, j] = float('inf')  # Facility failed
            elif (facility.id, demand.id) in scenario.route_blocks:
                costs[i, j] = float('inf')  # Route blocked
            else:
                actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
                costs[i, j] = calculate_transport_cost(facility, demand, actual_demand)
    transport_costs[scenario.id] = costs

print("Transport cost matrices calculated for all scenarios")
for scenario_id, costs in transport_costs.items():
    print(f"\nScenario {scenario_id} costs:")
    print(costs)

In [ ]:
# Formulate and solve the robust optimization problem
def solve_robust_network_design():
    """Solve the robust facility location problem using mixed-integer programming"""
    
    # Create the robust optimization model
    model = LpProblem("Resilient_Network_Design", LpMinimize)
    
    # First-stage decision variables: facility location decisions
    y = {}  # y[i] = 1 if facility i is built
    for i in range(len(facilities)):
        y[i] = LpVariable(f"y_{i}", cat='Binary')
    
    # Second-stage decision variables: flow from facilities to demand zones in each scenario
    x = {}  # x[i,j,s] = flow from facility i to demand zone j in scenario s
    z = {}  # z[j,s] = unmet demand at zone j in scenario s
    
    for s_idx, scenario in enumerate(scenarios):
        for i in range(len(facilities)):
            for j in range(len(demand_zones)):
                x[(i, j, s_idx)] = LpVariable(f"x_{i}_{j}_{s_idx}", lowBound=0)
        
        for j in range(len(demand_zones)):
            z[(j, s_idx)] = LpVariable(f"z_{j}_{s_idx}", lowBound=0)
    
    # Objective: minimize maximum total cost across all scenarios
    # We'll use an auxiliary variable for the maximum cost
    max_cost = LpVariable("max_cost", lowBound=0)
    
    # Calculate total cost for each scenario
    scenario_costs = []
    for s_idx, scenario in enumerate(scenarios):
        # Fixed costs for built facilities
        fixed_cost = lpSum([facilities[i].fixed_cost * y[i] for i in range(len(facilities))])
        
        # Transport costs
        transport_cost = lpSum([transport_costs[scenario.id][i, j] * x[(i, j, s_idx)] 
                               for i in range(len(facilities)) 
                               for j in range(len(demand_zones))
                               if transport_costs[scenario.id][i, j] != float('inf')])
        
        # Penalty for unmet demand (large M)
        M = 1000000  # Large penalty cost
        unmet_penalty = lpSum([M * z[(j, s_idx)] for j in range(len(demand_zones))])
        
        total_cost = fixed_cost + transport_cost + unmet_penalty
        scenario_costs.append(total_cost)
        
        # Constraint: scenario cost <= max_cost
        model += total_cost <= max_cost
    
    # Objective: minimize the maximum cost
    model += max_cost
    
    # Constraints for each scenario
    for s_idx, scenario in enumerate(scenarios):
        # Demand satisfaction constraints
        for j in range(len(demand_zones)):
            actual_demand = demand_zones[j].demand * scenario.demand_multipliers.get(demand_zones[j].id, 1.0)
            model += lpSum([x[(i, j, s_idx)] for i in range(len(facilities))]) + z[(j, s_idx)] == actual_demand
        
        # Capacity constraints
        for i in range(len(facilities)):
            if facilities[i].id not in scenario.facility_failures:
                model += lpSum([x[(i, j, s_idx)] for j in range(len(demand_zones))]) <= facilities[i].capacity * y[i]
            else:
                # Facility failed - no flow possible
                for j in range(len(demand_zones)):
                    model += x[(i, j, s_idx)] == 0
        
        # Route blocking constraints
        for (fac_id, demand_id) in scenario.route_blocks:
            fac_idx = fac_id - 1  # Convert to 0-based index
            demand_idx = demand_id - 1
            model += x[(fac_idx, demand_idx, s_idx)] == 0
    
    # Service coverage constraint: at least 80% in worst case
    # This is handled implicitly through the large penalty M for unmet demand
    
    # Solve the model
    print("Solving robust optimization model...")
    model.solve(PULP_CBC_CMD(msg=False))
    
    return model, y, x, z, max_cost

# Solve the problem
model, y_vars, x_vars, z_vars, max_cost_var = solve_robust_network_design()

In [ ]:
# Extract and analyze results
def analyze_results():
    """Analyze and display the optimization results"""
    
    print("=" * 60)
    print("ROBUST NETWORK DESIGN OPTIMIZATION RESULTS")
    print("=" * 60)
    
    # Check solution status
    print(f"\nSolution Status: {LpStatus[model.status]}")
    
    if model.status != LpStatusOptimal:
        print("No optimal solution found!")
        return
    
    # Extract facility decisions
    print("\nFACILITY LOCATION DECISIONS:")
    selected_facilities = []
    total_fixed_cost = 0
    
    for i in range(len(facilities)):
        if y_vars[i].value() > 0.5:  # Facility built
            selected_facilities.append(i)
            total_fixed_cost += facilities[i].fixed_cost
            print(f"  Facility {facilities[i].id}: BUILT (Cost: ${facilities[i].fixed_cost:,})")
        else:
            print(f"  Facility {facilities[i].id}: NOT BUILT")
    
    print(f"\nTotal Fixed Cost: ${total_fixed_cost:,}")
    
    # Analyze each scenario
    print("\n" + "=" * 60)
    print("SCENARIO ANALYSIS")
    print("=" * 60)
    
    worst_service_coverage = float('inf')
    worst_scenario_id = None
    
    for s_idx, scenario in enumerate(scenarios):
        print(f"\nScenario {scenario.id} ({'Normal' if scenario.id == 1 else 'Hurricane'}):")
        print(f"  Probability: {scenario.probability}")
        print(f"  Failed Facilities: {scenario.facility_failures}")
        print(f"  Blocked Routes: {scenario.route_blocks}")
        
        # Calculate total demand and served demand
        total_demand = 0
        served_demand = 0
        transport_cost = 0
        unmet_demand = 0
        
        for j in range(len(demand_zones)):
            actual_demand = demand_zones[j].demand * scenario.demand_multipliers.get(demand_zones[j].id, 1.0)
            total_demand += actual_demand
            
            zone_served = 0
            for i in range(len(facilities)):
                if x_vars[(i, j, s_idx)].value() is not None:
                    zone_served += x_vars[(i, j, s_idx)].value()
            
            served_demand += zone_served
            unmet_demand += z_vars[(j, s_idx)].value() or 0
            
            print(f"    Demand Zone {demand_zones[j].id}: {actual_demand:.1f} units, served: {zone_served:.1f}")
        
        # Calculate transport costs
        for i in range(len(facilities)):
            for j in range(len(demand_zones)):
                if x_vars[(i, j, s_idx)].value() is not None and transport_costs[scenario.id][i, j] != float('inf'):
                    transport_cost += transport_costs[scenario.id][i, j] * x_vars[(i, j, s_idx)].value()
        
        service_coverage = (served_demand / total_demand) * 100 if total_demand > 0 else 0
        total_scenario_cost = total_fixed_cost + transport_cost + unmet_demand * 1000000
        
        print(f"  Service Coverage: {service_coverage:.1f}%")
        print(f"  Transport Cost: ${transport_cost:,.0f}")
        print(f"  Unmet Demand Penalty: ${unmet_demand * 1000000:,.0f}")
        print(f"  Total Scenario Cost: ${total_scenario_cost:,.0f}")
        
        # Track worst-case scenario
        if service_coverage < worst_service_coverage:
            worst_service_coverage = service_coverage
            worst_scenario_id = scenario.id
    
    print(f"\n" + "=" * 60)
    print("WORST-CASE ANALYSIS")
    print("=" * 60)
    print(f"Worst-Case Service Coverage: {worst_service_coverage:.1f}% (Scenario {worst_scenario_id})")
    print(f"Maximum Total Cost: ${max_cost_var.value():,.0f}")
    
    # Check if solution meets requirements
    if worst_service_coverage >= 80:
        print("✓ SOLUTION MEETS 80% SERVICE COVERAGE REQUIREMENT")
    else:
        print("✗ SOLUTION DOES NOT MEET 80% SERVICE COVERAGE REQUIREMENT")
    
    return selected_facilities, worst_service_coverage, max_cost_var.value()

# Analyze the results
selected_facilities, worst_coverage, max_cost = analyze_results()

In [ ]:
# Create comprehensive visualization
def create_visualization():
    """Create a comprehensive 4-panel visualization of the results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Robust Network Design Optimization Results', fontsize=16, fontweight='bold')
    
    # Panel 1: Network Layout
    ax1 = axes[0, 0]
    ax1.set_title('Network Layout and Facility Selection', fontweight='bold')
    
    # Plot all facilities
    for i, facility in enumerate(facilities):
        color = 'green' if i in selected_facilities else 'red'
        marker = 'o' if i in selected_facilities else 'x'
        size = 300 if i in selected_facilities else 200
        ax1.scatter(facility.x, facility.y, c=color, marker=marker, s=size, 
                   alpha=0.7, edgecolors='black', linewidth=2)
        ax1.annotate(f'F{facility.id}', (facility.x, facility.y), 
                    xytext=(5, 5), textcoords='offset points', fontweight='bold')
    
    # Plot demand zones
    for demand in demand_zones:
        ax1.scatter(demand.x, demand.y, c='blue', marker='s', s=200, 
                   alpha=0.7, edgecolors='black', linewidth=2)
        ax1.annotate(f'D{demand.id}', (demand.x, demand.y), 
                    xytext=(5, 5), textcoords='offset points', fontweight='bold')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.legend(['Selected Facility', 'Rejected Facility', 'Demand Zone'], 
               loc='upper right', bbox_to_anchor=(1.15, 1))
    
    # Panel 2: Scenario Comparison
    ax2 = axes[0, 1]
    ax2.set_title('Scenario Performance Comparison', fontweight='bold')
    
    scenario_names = ['Normal', 'Hurricane']
    service_coverages = []
    transport_costs = []
    
    for s_idx, scenario in enumerate(scenarios):
        # Calculate service coverage
        total_demand = 0
        served_demand = 0
        for j in range(len(demand_zones)):
            actual_demand = demand_zones[j].demand * scenario.demand_multipliers.get(demand_zones[j].id, 1.0)
            total_demand += actual_demand
            for i in range(len(facilities)):
                if x_vars[(i, j, s_idx)].value() is not None:
                    served_demand += x_vars[(i, j, s_idx)].value()
        
        service_coverage = (served_demand / total_demand) * 100 if total_demand > 0 else 0
        service_coverages.append(service_coverage)
        
        # Calculate transport cost
        transport_cost = 0
        for i in range(len(facilities)):
            for j in range(len(demand_zones)):
                if x_vars[(i, j, s_idx)].value() is not None and transport_costs[scenario.id][i, j] != float('inf'):
                    transport_cost += transport_costs[scenario.id][i, j] * x_vars[(i, j, s_idx)].value()
        transport_costs.append(transport_cost)
    
    x_pos = np.arange(len(scenario_names))
    width = 0.35
    
    bars1 = ax2.bar(x_pos - width/2, service_coverages, width, label='Service Coverage (%)', alpha=0.8)
    ax2_twin = ax2.twinx()
    bars2 = ax2_twin.bar(x_pos + width/2, transport_costs, width, label='Transport Cost ($)', alpha=0.8, color='orange')
    
    ax2.set_xlabel('Scenario')
    ax2.set_ylabel('Service Coverage (%)', color='blue')
    ax2_twin.set_ylabel('Transport Cost ($)', color='orange')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(scenario_names)
    ax2.grid(True, alpha=0.3)
    
    # Add 80% service coverage threshold line
    ax2.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% Threshold')
    
    # Panel 3: Cost Breakdown
    ax3 = axes[1, 0]
    ax3.set_title('Cost Structure Analysis', fontweight='bold')
    
    # Calculate costs for each scenario
    fixed_costs = [sum(facilities[i].fixed_cost for i in selected_facilities)] * len(scenarios)
    
    total_costs = []
    for s_idx, scenario in enumerate(scenarios):
        transport_cost = 0
        unmet_penalty = 0
        for j in range(len(demand_zones)):
            unmet_penalty += (z_vars[(j, s_idx)].value() or 0) * 1000000
        for i in range(len(facilities)):
            for j in range(len(demand_zones)):
                if x_vars[(i, j, s_idx)].value() is not None and transport_costs[scenario.id][i, j] != float('inf'):
                    transport_cost += transport_costs[scenario.id][i, j] * x_vars[(i, j, s_idx)].value()
        total_costs.append(fixed_costs[s_idx] + transport_cost + unmet_penalty)
    
    width = 0.25
    x_pos = np.arange(len(scenario_names))
    
    bars1 = ax3.bar(x_pos - width, fixed_costs, width, label='Fixed Costs', alpha=0.8)
    bars2 = ax3.bar(x_pos, transport_costs, width, label='Transport Costs', alpha=0.8)
    bars3 = ax3.bar(x_pos + width, [tc - fc - tr for tc, fc, tr in zip(total_costs, fixed_costs, transport_costs)], 
                    width, label='Unmet Demand Penalties', alpha=0.8)
    
    ax3.set_xlabel('Scenario')
    ax3.set_ylabel('Cost ($)')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(scenario_names)
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Sensitivity Analysis
    ax4 = axes[1, 1]
    ax4.set_title('Service Coverage Sensitivity', fontweight='bold')
    
    # Create sensitivity data by varying service requirements
    service_requirements = [70, 75, 80, 85, 90, 95]
    costs_at_requirements = []
    
    for req in service_requirements:
        # This is a simplified sensitivity analysis
        # In practice, we would re-solve the model for each requirement
        if req <= worst_coverage:
            costs_at_requirements.append(max_cost)
        else:
            # Estimate higher cost for higher requirements
            cost_increase = (req - worst_coverage) * 50000  # $50k per percentage point
            costs_at_requirements.append(max_cost + cost_increase)
    
    ax4.plot(service_requirements, costs_at_requirements, 'bo-', linewidth=2, markersize=8)
    ax4.axhline(y=max_cost, color='red', linestyle='--', alpha=0.7, label=f'Current Solution: ${max_cost:,.0f}')
    ax4.axvline(x=80, color='green', linestyle='--', alpha=0.7, label='80% Requirement')
    ax4.axvline(x=worst_coverage, color='orange', linestyle='--', alpha=0.7, label=f'Achieved: {worst_coverage:.1f}%')
    
    ax4.set_xlabel('Service Coverage Requirement (%)')
    ax4.set_ylabel('Total Cost ($)')
    ax4.grid(True, alpha=0.3)
    ax4.legend(loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualization
fig = create_visualization()

### Why this Tier exists vs earlier Tiers
This Tier 1 provides the mathematical foundation for resilient network design:
- **Exact optimization**: Provides provably optimal solutions for the robust facility location problem
- **Comprehensive modeling**: Handles multiple uncertainty scenarios, facility failures, route blockages, and demand shifts
- **Theoretical benchmark**: Serves as the gold standard against which heuristic and metaheuristic approaches can be compared
- **Decision transparency**: Every constraint and objective is explicitly formulated and interpretable

### Pros / Cons vs other approaches
**Pros:**
- Guaranteed optimality for the given problem formulation
- Complete consideration of all constraints and scenarios
- Provides a baseline for evaluating approximate methods
- Transparent decision-making process

**Cons:**
- Computational complexity grows exponentially with problem size
- May become intractable for large-scale networks
- Requires precise problem data and parameters
- Limited flexibility for dynamic or real-time adjustments

### When to use this Tier
- **Small to medium-sized networks** where exact solutions are feasible
- **Strategic planning** decisions where optimality is critical
- **Benchmark development** for evaluating other solution methods
- **Regulatory or compliance** situations requiring provable optimal solutions
- **Academic research** and theoretical analysis of resilient network design